# XNES vs CMA-ES on COCO BBOB

This notebook runs three optimizers on the full COCO `bbob` suite with COCO-style budgeting.

- CMA-ES: `cma.CMAEvolutionStrategy` defaults (except `sigma0`, which CMA requires explicitly)
- XNES: `Optimizer()` with `csa_enabled=False`
- XNES + CSA: `Optimizer()` with `csa_enabled=True` (made explicit even though this is the default)
- Budget per problem instance: `budget_multiplier * dimension`


In [ ]:
from __future__ import annotations

from pathlib import Path
import warnings

import cma
import cocoex
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from leitwerk import Optimizer

warnings.filterwarnings('ignore', category=RuntimeWarning)


In [ ]:
# COCO experiment settings (algorithm settings remain defaults).
SUITE_NAME = 'bbob-noisy'
SUITE_INSTANCE = ''
SUITE_OPTIONS = ''
BUDGET_MULTIPLIER = 1000
SIGMA0_CMA = 1.0  # required constructor arg for CMA-ES

OUTPUT_ROOT = Path('exdata')  # COCO writes under this root automatically


In [ ]:
def run_xnes(problem: cocoex.Problem, budget: int, *, csa_enabled: bool) -> None:
    x0 = np.asarray(problem.initial_solution, dtype=float)
    optimizer = Optimizer()
    optimizer.csa_enabled = csa_enabled
    for i, v in enumerate(x0):
        optimizer.add(f'x{i:04d}', loc=float(v))

    while problem.evaluations < budget and not problem.final_target_hit:
        x = np.asarray([param.value for param in params], dtype=float)
        value = float(problem(x))
        optimizer.tell(-value)


def run_cma(problem: cocoex.Problem, budget: int, sigma0: float = SIGMA0_CMA) -> None:
    x0 = np.asarray(problem.initial_solution, dtype=float)
    es = cma.CMAEvolutionStrategy(x0, sigma0)

    while not es.stop() and problem.evaluations < budget and not problem.final_target_hit:
        remaining = budget - problem.evaluations
        if remaining < es.popsize:
            break
        candidates = es.ask()
        values = [float(problem(x)) for x in candidates]
        es.tell(candidates, values)


In [ ]:
def benchmark(label: str, runner) -> pd.DataFrame:
    suite = cocoex.Suite(SUITE_NAME, SUITE_INSTANCE, SUITE_OPTIONS)
    observer = cocoex.Observer(SUITE_NAME, f'result_folder: {label}')
    rows = []

    for problem in suite:
        problem.observe_with(observer)
        budget = BUDGET_MULTIPLIER * problem.dimension
        start_evals = int(problem.evaluations)

        runner(problem, budget)

        rows.append({
            'optimizer': label,
            'problem_id': problem.id,
            'function': int(problem.id_function),
            'instance': int(problem.id_instance),
            'dimension': int(problem.dimension),
            'evaluations': int(problem.evaluations - start_evals),
            'budget': int(budget),
            'target_hit': bool(problem.final_target_hit),
            'best_f': float(problem.best_observed_fvalue1),
        })

        problem.free()

    suite.free()
    return pd.DataFrame(rows)


In [ ]:
cma_df = benchmark('cma', run_cma)
xnes_df = benchmark('xnes', lambda problem, budget: run_xnes(problem, budget, csa_enabled=False))
xnes_csa_df = benchmark('xnes_csa', lambda problem, budget: run_xnes(problem, budget, csa_enabled=True))
results = pd.concat([cma_df, xnes_df, xnes_csa_df], ignore_index=True)

results.head()


In [ ]:
summary = (
    results.groupby(['optimizer', 'dimension'], as_index=False)
    .agg(
        success_rate=('target_hit', 'mean'),
        median_best_f=('best_f', 'median'),
        median_evals=('evaluations', 'median'),
        n=('problem_id', 'count'),
    )
)
summary


In [ ]:
pivot = summary.pivot(index='dimension', columns='optimizer', values='success_rate').sort_index()
ax = pivot.plot(marker='o', figsize=(8, 4), title='BBOB Success Rate by Dimension')
ax.set_ylabel('success rate')
ax.set_xlabel('dimension')
ax.grid(True, alpha=0.3)
plt.show()


In [ ]:
# Optional COCO post-processing report (HTML)
import cocopp
cocopp.main([str(OUTPUT_ROOT / 'cma'), str(OUTPUT_ROOT / 'xnes'), str(OUTPUT_ROOT / 'xnes_csa')])
